# Interactive Chronos Inference & Plotting
This notebook lets you pick a specific series from the test set, run inference on it using either the base model or the fine-tuned model, and visualize the output instantly.


In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure we can import chronos
WORK_ROOT = os.path.abspath('..')
if WORK_ROOT not in sys.path:
    sys.path.append(WORK_ROOT)

from chronos import BaseChronosPipeline
from chronos.chronos2 import Chronos2Pipeline
from transformers.utils.peft_utils import find_adapter_config_file


## 1. Discover Available Series


In [ ]:
TEST_PKL_PATH = "./test_model_inputs.pkl"

if not os.path.exists(TEST_PKL_PATH):
    print(f"Error: Could not find {TEST_PKL_PATH}.")
else:
    with open(TEST_PKL_PATH, "rb") as f:
        all_windows = pickle.load(f)
        
    unique_series = sorted(list(set([w["series_id"] for w in all_windows])))
    print("Available Series for Inference & Plotting:")
    for s in unique_series:
        print(f" - {s}")


## 2. Configuration
Set your desired series and which model to inference with.


In [ ]:
# ==========================================
# 1. Select the Series to evaluate
# ==========================================
SERIES_ID = "SMD_machine-2-5_test.csv"

# ==========================================
# 2. Select Model Checkpoint
# ==========================================
# Options: "finetuned" or "base"
MODEL_TYPE = "finetuned"

# Paths to the models
BASE_MODEL_ID = "amazon/chronos-2"
FINETUNED_CKPT = "/home/rajib/Sir_git_TSAD/TSFM-anomaly/Chronos_Finetuning/rajib_work_space/SMD_run/chronos2-single-stage_SMD/finetuned-ckpt"

# ==========================================
# 3. Evaluation & Plotting Settings
# ==========================================
NORMAL_SIGNAL_LENGTH = 256
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 64
PREDICT_BATCH_SIZE = 128

# Inference now runs for ALL variables in the series (multivariate, like forward.py),
# so every feature is available to plot.
PLOT_ALL_FEATURES = True   # True -> one figure per variable; False -> only FEATURE_IDX
FEATURE_IDX = 0            # Which single feature to plot when PLOT_ALL_FEATURES = False
PLOT_FROM = None          # Start index for zoom (e.g., 1440)
PLOT_UNTIL = None         # End index for zoom (e.g., 2040)
SMOOTH_WINDOW = 100       # Smoothing window (set to 1 for no smoothing)


## 3. Run Inference
This cell filters down the dataset to only your selected series and runs the Chronos pipeline.


In [ ]:
# Filter windows for just our chosen series
target_windows = [w for w in all_windows if w["series_id"] == SERIES_ID]
print(f"Found {len(target_windows)} windows for series '{SERIES_ID}'.")

# Load per-series ground-truth meta (gives the full series length + labels so we
# can reassemble the per-window forecasts onto the original timeline).
META_PKL_PATH = "test_series_meta.pkl"
with open(META_PKL_PATH, "rb") as f:
    meta_data = pickle.load(f)

if len(target_windows) == 0:
    print(f"Warning: No windows found for {SERIES_ID}")
else:
    # Setup Model
    model_path = FINETUNED_CKPT if MODEL_TYPE == "finetuned" else BASE_MODEL_ID
    print(f"Loading {MODEL_TYPE.upper()} model from: {model_path} ...")

    # We clear cache to free up VRAM from any previous cell executions
    torch.cuda.empty_cache()

    if find_adapter_config_file(model_path) is not None:
        pipeline = Chronos2Pipeline.from_pretrained(model_path, device_map="cuda", torch_dtype=torch.bfloat16)
    else:
        pipeline = BaseChronosPipeline.from_pretrained(model_path, device_map="cuda", torch_dtype=torch.bfloat16)

    use_sep = bool(getattr(pipeline.model.chronos_config, "use_sep_token", False))
    if use_sep and MODEL_TYPE == "finetuned":
        print("Restoring SEP token patch index (16) for fine-tuned adapter...")
        pipeline.model.chronos_config.sep_patch_index = int(NORMAL_SIGNAL_LENGTH / 16)

    use_normal_prefix = True if MODEL_TYPE == "finetuned" else False

    # Chronos-2 predict() returns QUANTILES, not raw samples. We feed the FULL
    # multivariate target for each window (all variables at once, exactly like
    # forward.py), so each returned element has shape (n_variates, n_quantiles, P).
    # Look up the q10/q50/q90 columns from the model config and index that axis.
    quantiles = list(pipeline.model.chronos_config.quantiles)
    qi10, qi50, qi90 = quantiles.index(0.1), quantiles.index(0.5), quantiles.index(0.9)

    # Per-feature arrays over the series' ORIGINAL global timeline (NaN = uncovered).
    # Shape (n_features, series_len); windows are scattered by future_start/future_end.
    n_features = int(meta_data[SERIES_ID]["n_features"])
    series_len = int(meta_data[SERIES_ID]["length"])
    full_actual = np.full((n_features, series_len), np.nan)
    full_q10 = np.full((n_features, series_len), np.nan)
    full_q50 = np.full((n_features, series_len), np.nan)
    full_q90 = np.full((n_features, series_len), np.nan)

    in_lo = 0 if use_normal_prefix else NORMAL_SIGNAL_LENGTH     # feed [normal|context] or [context]
    fut_lo = NORMAL_SIGNAL_LENGTH + CONTEXT_LENGTH               # start of the future slice
    fut_hi = fut_lo + PREDICTION_LENGTH
    model_context = fut_lo - in_lo                               # input length (768 with prefix)

    total_len = len(target_windows)
    print(f"Running multivariate inference for {n_features} variables ...")

    for i in range(0, total_len, PREDICT_BATCH_SIZE):
        batch = target_windows[i : i + PREDICT_BATCH_SIZE]
        # Each input is the full (n_variates, model_context) target -> the model
        # forecasts all variables jointly for this window.
        inputs = [{"target": w["target"][:, in_lo:fut_lo]} for w in batch]

        print(f"Predicting batch {i} to {i + len(batch)} ...")
        forecast = pipeline.predict(
            inputs,
            prediction_length=PREDICTION_LENGTH,
            context_length=model_context,   # keep full input -> SEP stays aligned
        )

        for w, pred in zip(batch, forecast):
            pred = pred.float().cpu().numpy()        # (n_variates, n_quantiles, P)
            q10 = pred[:, qi10, :]                   # (n_variates, P)
            q50 = pred[:, qi50, :]
            q90 = pred[:, qi90, :]
            act = np.asarray(w["target"][:, fut_lo:fut_hi], dtype=float)   # (n_variates, P)

            fs, fe = int(w["future_start"]), int(w["future_end"])
            n = min(fe, series_len) - fs             # guard against running past the series end
            full_actual[:, fs:fs + n] = act[:, :n]
            full_q10[:, fs:fs + n] = q10[:, :n]
            full_q50[:, fs:fs + n] = q50[:, :n]
            full_q90[:, fs:fs + n] = q90[:, :n]

    score = (full_actual - full_q50) ** 2   # per-feature MSE anomaly score (n_features, series_len)

    print(f"\nInference complete! Arrays have shape {full_actual.shape} (n_features, series_len).")


## 4. Plot Results


In [ ]:
%matplotlib inline


def plot_inference(series_id, full_actual, full_q10, full_q50, full_q90, score,
                   meta_data, feature_idx=0, model_type="finetuned",
                   plot_from=None, plot_until=None, smooth_window=1):
    """Plot ONE variable: actual vs. predicted (with the 0.1-0.9 prediction
    interval) on top, and THAT feature's own anomaly score against the series'
    ground-truth labels below.

    `full_actual`, `full_q10`, `full_q50`, `full_q90`, `score` are all
    (n_features, series_len) arrays laid out on the series' global timeline
    (NaN where uncovered); we select row `feature_idx`. The anomaly score is
    feature-wise: score[feature_idx] = (actual - q50) ** 2 for this feature only.

    Returns the matplotlib Figure.
    """
    # Select this single feature's row from each (n_features, series_len) array.
    actuals = np.asarray(full_actual, dtype=float)[feature_idx]
    q10 = np.asarray(full_q10, dtype=float)[feature_idx]
    q50 = np.asarray(full_q50, dtype=float)[feature_idx]
    q90 = np.asarray(full_q90, dtype=float)[feature_idx]
    anomaly_score = np.asarray(score, dtype=float)[feature_idx]   # feature-wise score

    # Ground-truth labels (full per-timestamp), already on the same timeline.
    if series_id in meta_data and "labels" in meta_data[series_id]:
        ground_truth = np.asarray(meta_data[series_id]["labels"], dtype=float)
    else:
        ground_truth = np.zeros_like(anomaly_score)

    # Align everything to the shortest common length.
    min_len = min(len(actuals), len(ground_truth))
    actuals, q10, q50, q90 = actuals[:min_len], q10[:min_len], q50[:min_len], q90[:min_len]
    anomaly_score, ground_truth = anomaly_score[:min_len], ground_truth[:min_len]

    # Scale THIS feature's anomaly score to [0, 1], ignoring NaN (uncovered region).
    y_min, y_max = np.nanmin(anomaly_score), np.nanmax(anomaly_score)
    denom = y_max - y_min if y_max != y_min else 1.0
    y_score_scaled = (anomaly_score - y_min) / denom

    if smooth_window > 1:
        def _roll(a):
            return pd.Series(a).rolling(smooth_window, center=True, min_periods=1).mean().values
        q50, q10, q90 = _roll(q50), _roll(q10), _roll(q90)
        y_score_scaled = _roll(y_score_scaled)

    time_steps = np.arange(min_len)
    start = plot_from if plot_from is not None else 0
    end = plot_until if plot_until is not None else min_len
    sl = slice(start, end)
    time_steps = time_steps[sl]
    actuals, q10, q50, q90 = actuals[sl], q10[sl], q50[sl], q90[sl]
    y_score_scaled, ground_truth = y_score_scaled[sl], ground_truth[sl]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 6),
                                   gridspec_kw={'height_ratios': [4, 1]}, sharex=True)
    fig.suptitle(f" Series: {series_id} | Feature: {feature_idx} | Model: {model_type.upper()} ",
                 fontsize=10, fontweight='bold')

    ax1.plot(time_steps, actuals, label='Actual', color='steelblue', linewidth=1.2)
    ax1.plot(time_steps, q50, label='Predicted', color='darkorange', linestyle='--', linewidth=1.2)
    ax1.fill_between(time_steps, q10, q90, color='xkcd:light lavender', alpha=0.3,
                     label='prediction interval (0.1-0.9)')
    ax1.set_ylabel(f'Value (feat {feature_idx})')
    ax1.legend(loc='upper left')
    ax1.grid(True, linestyle='--', alpha=0.4)

    ax2.plot(time_steps, y_score_scaled, label=f'anomaly score (feat {feature_idx})',
             color='steelblue', linewidth=1.0)
    ax2.fill_between(time_steps, y_score_scaled, alpha=0.15, color='steelblue')
    ax2.fill_between(time_steps, 0, 1, where=(ground_truth == 1), color='red', alpha=0.3,
                     label='is_anomaly (ground truth)', step='mid')
    ax2.plot(time_steps, ground_truth, color='red', linewidth=0.8, drawstyle='steps-mid', alpha=0.6)
    ax2.set_ylabel('Score / Anomaly')
    ax2.set_ylim(-0.1, 1.5)
    ax2.set_yticks([0, 0.5, 1])
    ax2.set_xlabel('timestamp')
    ax2.legend(loc='upper left')
    ax2.grid(True, linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()
    return fig


if len(target_windows) > 0:
    n_features = full_actual.shape[0]
    # Iterate through ALL variables (PLOT_ALL_FEATURES=True) -> one figure per
    # feature, each showing that feature's OWN anomaly score; or just FEATURE_IDX.
    features_to_plot = list(range(n_features)) if PLOT_ALL_FEATURES else [FEATURE_IDX]
    print(f"Plotting {len(features_to_plot)} feature(s) "
          f"({'all variables' if PLOT_ALL_FEATURES else f'feature {FEATURE_IDX}'}) "
          f"-- each panel shows that feature's own anomaly score.")

    for f in features_to_plot:
        plot_inference(
            SERIES_ID, full_actual, full_q10, full_q50, full_q90, score,
            meta_data, feature_idx=f, model_type=MODEL_TYPE,
            plot_from=PLOT_FROM, plot_until=PLOT_UNTIL, smooth_window=SMOOTH_WINDOW,
        )
